# Module 6 / Class 5 -- LSTM vs Logistic Regression on IMDB

**Objectives:**
- Build a TF-IDF + Logistic Regression baseline for sentiment analysis
- Build an LSTM model for the same task
- Compare both approaches on accuracy

**Runtime:** Use GPU for faster LSTM training.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print(f"TensorFlow version: {tf.__version__}")

## 1. Load IMDB Dataset

In [ ]:
# Load IMDB with a vocabulary of 10,000 most frequent words
VOCAB_SIZE = 10000
MAXLEN = 200  # truncate/pad reviews to this length

(X_train_seq, y_train), (X_test_seq, y_test) = keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)

print(f"Training samples: {len(X_train_seq)}")
print(f"Test samples:     {len(X_test_seq)}")
print(f"Label distribution (train): {np.bincount(y_train)}")
print(f"\nExample review (as integer sequence, first 20 tokens):")
print(X_train_seq[0][:20])

In [ ]:
# Decode a review back to text for illustration
word_index = keras.datasets.imdb.get_word_index()
reverse_index = {v + 3: k for k, v in word_index.items()}
reverse_index[0] = '<PAD>'
reverse_index[1] = '<START>'
reverse_index[2] = '<UNK>'
reverse_index[3] = '<UNUSED>'

decoded = ' '.join(reverse_index.get(i, '?') for i in X_train_seq[0][:50])
print(f"Decoded (first 50 tokens): {decoded}")
print(f"Label: {'Positive' if y_train[0] == 1 else 'Negative'}")

In [ ]:
# Review length distribution
lengths = [len(x) for x in X_train_seq]
plt.figure(figsize=(8, 4))
plt.hist(lengths, bins=50, edgecolor='black', color='steelblue')
plt.axvline(x=MAXLEN, color='red', linestyle='--', label=f'MAXLEN={MAXLEN}')
plt.title('Review Length Distribution')
plt.xlabel('Number of Tokens')
plt.ylabel('Count')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print(f"Mean length: {np.mean(lengths):.0f}, Median: {np.median(lengths):.0f}")

## 2. Baseline -- TF-IDF + Logistic Regression

We convert integer sequences back to text strings so we can use TfidfVectorizer.

In [ ]:
# Convert sequences back to text for TF-IDF
def sequences_to_texts(sequences):
    return [' '.join(reverse_index.get(i, '') for i in seq) for seq in sequences]

train_texts = sequences_to_texts(X_train_seq)
test_texts = sequences_to_texts(X_test_seq)

print(f"Sample text (first 100 chars): {train_texts[0][:100]}")

In [ ]:
# TF-IDF vectorization
tfidf = TfidfVectorizer(max_features=VOCAB_SIZE)
X_train_tfidf = tfidf.fit_transform(train_texts)
X_test_tfidf = tfidf.transform(test_texts)

print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")

In [ ]:
# Logistic Regression
lr_model = LogisticRegression(max_iter=1000, C=1.0)
lr_model.fit(X_train_tfidf, y_train)

lr_train_acc = accuracy_score(y_train, lr_model.predict(X_train_tfidf))
lr_test_acc = accuracy_score(y_test, lr_model.predict(X_test_tfidf))

print(f"Logistic Regression -- Train accuracy: {lr_train_acc:.4f}")
print(f"Logistic Regression -- Test accuracy:  {lr_test_acc:.4f}")

## 3. LSTM Model

Architecture: Embedding -> LSTM -> Dense(1, sigmoid)

In [ ]:
# Pad sequences to fixed length for LSTM
X_train_pad = keras.preprocessing.sequence.pad_sequences(X_train_seq, maxlen=MAXLEN)
X_test_pad = keras.preprocessing.sequence.pad_sequences(X_test_seq, maxlen=MAXLEN)

print(f"Padded shape: {X_train_pad.shape}")

In [ ]:
EMBEDDING_DIM = 64
LSTM_UNITS = 64

lstm_model = keras.Sequential([
    layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAXLEN),
    layers.LSTM(LSTM_UNITS),
    layers.Dense(1, activation='sigmoid'),
])

lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

lstm_model.summary()

In [ ]:
lstm_history = lstm_model.fit(
    X_train_pad, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1,
)

In [ ]:
lstm_test_loss, lstm_test_acc = lstm_model.evaluate(X_test_pad, y_test, verbose=0)
print(f"LSTM -- Test accuracy: {lstm_test_acc:.4f}")

## 4. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(lstm_history.history['accuracy'], label='Train', linewidth=2)
axes[0].plot(lstm_history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0].set_title('LSTM -- Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(lstm_history.history['loss'], label='Train', linewidth=2)
axes[1].plot(lstm_history.history['val_loss'], label='Validation', linewidth=2)
axes[1].set_title('LSTM -- Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Final Comparison

In [ ]:
print("=" * 45)
print(f"{'Model':<25} {'Test Accuracy':>15}")
print("=" * 45)
print(f"{'TF-IDF + LogReg':<25} {lr_test_acc:>15.4f}")
print(f"{'LSTM':<25} {lstm_test_acc:>15.4f}")
print("=" * 45)

---

## TODO: Student Work -- Comparison Analysis

Write a short analysis comparing the two models. Address:

1. Which model performed better on test accuracy?
2. Which model trained faster?
3. What are the strengths and weaknesses of each approach?
4. In what scenarios would you prefer TF-IDF + LogReg over LSTM, and vice versa?
5. What could you do to improve the LSTM model?

# TODO: Write your comparison analysis

1. Performance: ...
2. Training speed: ...
3. Strengths/weaknesses: ...
4. When to use which: ...
5. LSTM improvements: ...